# Zeros and Zero-Crossings of a Curated Spline
We synthesize a spline out of curated spline coefficients. We display this spline in <span style="color:#1f77b4">**blue**</span>, with blue stems and rings that highlight the samples at the integers, and red stems and rings that highlight the samples at the boundaries of one period. The black dots give the knots of the spline.

We then extract from this curated spline the list of the intervals where the spline vanishes, which we overlay as thick lines and markers in the <span style="color:#0343df">**zaffer**</span> color. Moreover, we also give access to the zero-crossings of the curated spline. Finally, we print a verbose description of either the zeros or the zero-crossings.

In [1]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Curated spline
f = sk.PeriodicSpline1D.from_spline_coeff(
    np.array([-5.0, 5.0, 4.0, 5.0, 0.0, 0.0, 6.0, -5.0 / 3.0, 23.0 / 15.0]),
    degree = 1
)

# Plot
def update_plot (
    degree = 1,
    zerotype = 0
):
    # Update the spline while maintaining the spline coefficients
    f.degree = degree
    # Plot the spline
    (fig, ax) = plt.subplots()
    f.plot((fig, ax), plotpoints = 301)

    # Zero pieces
    zeros = []
    displayed_zeros = []
    if 0 == zerotype: # Zeros
        zeros = f.zeros()
        displayed_zeros = zeros
    elif zerotype in {1, 2, 3}: # Zero-crossings
        zeros = f.zero_crossings()
        if 1 == zerotype: # All zero-crossings
            displayed_zeros = zeros[0] + zeros[1]
        elif 2 == zerotype: # Descending zero-crossings
            displayed_zeros = zeros[0]
        elif 3 == zerotype: # Ascending zero-crossings
            displayed_zeros = zeros[1]

    # Plot each piece independently
    for z in displayed_zeros:
        lb = z.infimum # Lower bound of the domain
        ub = z.supremum # Upper bound of the domain
        if not z.isleftopen:
            ax.plot(
                lb,
                0.0,
                marker = "o",
                markerfacecolor = "b",
                markeredgecolor = "b",
                markersize = 7.0
            )
        if not z.isrightopen:
            ax.plot(
                ub,
                0.0,
                marker = "o",
                markerfacecolor = "b",
                markeredgecolor = "b",
                markersize = 7.0
            )
        if lb != ub:
            ax.plot([lb, ub], [0.0, 0.0], "-b", linewidth = 3.0)
    # Show the plot
    plt.show()

    # Verbose description
    print("---")
    if 0 == zerotype: # Zeros
        print("# Zeros")
        for z in zeros:
            print(z)
    if zerotype in {1, 2}: # Descending zero-crossings
        print("# Descending zero-crossings")
        for z in zeros[0]:
            print(z)
    if zerotype in {1, 3}: # Ascending zero-crossings
        print("# Ascending zero-crossings")
        for z in zeros[1]:
            print(z)

# Interactions
degree_toggle_buttons_widget = widgets.ToggleButtons(
    options = [
        ("Zero", 0),
        ("Linear", 1),
        ("Quadratic", 2),
        ("Cubic", 3),
        ("Quartic", 4),
        ("Quintic", 5)
    ],
    value = 1,
    description = "degree:",
    style = widgets.widget_selection.ToggleButtonsStyle(button_width = "7em")
)
zerotype_radio_buttons_widget = widgets.RadioButtons(
    options = [
        ("Zeros", 0),
        ("0-Xings", 1), 
        ("\u2193 0-Xings", 2),
        ("\u2191 0-Xings", 3)
    ],
    value = 0,
    description = "type:"
)
widgets.interactive(
    update_plot,
    degree = degree_toggle_buttons_widget,
    zerotype = zerotype_radio_buttons_widget
)

interactive(children=(ToggleButtons(description='degree:', index=1, options=(('Zero', 0), ('Linear', 1), ('Qua…

## What to Observe
### Linear
Focus on the zeros within the open interval $(4,5)$ which, complemented by the two pointwise zeroes found at the degenerate intervals $[4,4]$ and $[5,5]$, result in the *closed* interval $[4,5]$ over which the linear spline vanishes. This interval is a zero of the spline, but it is not a zero-crossing.

Now, reduce the spline degree to zero.

### Zero
Near the closed interval $[4,5]$ where the linear spline was vanishing, the spline of degree zero has zeros over the interval $(3+1/2,5+1/2)$ which, as it turns out, is now an *open* interval. Still, there are no zero-crossings there.

The zero-degree spline also takes value zero at argument $1/2,$ an argument where the spline takes some value on one side of the discontinuity and the same value up to sign on the other side. At other discontinuities, for instance at $6+1/2,$ it can also happen that the spline straddles zero; however, the `splinekit.PeriodicSpline1D.zeros()` function reports only those arguments where the spline itself takes value zero, which happens at spline discontinuities only where the two sides of the discontinuity take identical absolute values. It is instructive to compare this state of affairs to that of `splinekit.PeriodicSpline1D.zero_crossings()`.

Now, augment the spline degree to make it quadratic and choose to display its zeros.

### Quadratic
In addition to four locations where square-free polynomial pieces of the spline cross the horizontal axis, observe that there exists a fifth one, somewhere near $4+1/2,$ where the library asserts that the spline vanishes, too. However, numerical inaccuracies result in a tiny interval being returned rather than the exact value $4+1/2$ predicted by the arithmetic analysis of the values of the spline coefficients. Yet, no zero-crossing is detected there.

Finally, augment the spline once more to make it cubic.

### Cubic
Observe that three clustered zeroes are found over an open interval of unit length that contains no knot, specifically after the spline knot at argument $7$ and before the spline knot at argument $8.$

The total number of zeros is four; it matches the total number of zero-crossings. While the diameter of the enclosure of either the zeros or the zero-crossings is guaranteed to not exceed one spline period, the respective enclosures are not necessarily taken at the same location.